# 부품 자동추천 AI — Colab 파이프라인 (수집 + QLoRA 학습)

Google Colab **Pro (L4 24GB / A100 권장)** 에서 데이터 수집부터 Qwen3-8B QLoRA 학습·추론까지 한 번에 실행합니다.

1. GPU/환경 확인 → 2. Google Drive 마운트 → 3. 코드/의존성 설치 → 4. (수집용) Ollama 기동 →
5. 시크릿 설정 → 6. 데이터 수집 → 7. 전처리 → 8. QLoRA 학습 → 9. 추론 테스트

> **런타임 → 런타임 유형 변경**에서 GPU(L4/A100)를 선택했는지 먼저 확인하세요. Colab 런타임은 휘발성이므로 데이터/어댑터는 모두 Google Drive에 저장합니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. Google Drive 마운트
수집 데이터·학습 산출물(어댑터)을 Drive에 영구 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/auto_search'   # Drive 내 저장 위치
os.makedirs(f'{DRIVE_ROOT}/datasheets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
print('Drive 저장 경로:', DRIVE_ROOT)

## 3. 코드 가져오기 + 의존성 설치
아래 `REPO_URL`을 본인 저장소 주소로 바꾸세요. (또는 Drive에 올려둔 코드를 사용)

In [ ]:
REPO_URL = 'https://github.com/<YOUR_ID>/auto_search.git'   # ← 수정
%cd /content
![ -d auto_search ] || git clone $REPO_URL
%cd /content/auto_search

# 데이터시트 PDF 파싱용 Java (OpenDataLoader 는 JVM 기반)
!apt-get -qq install -y default-jre > /dev/null && java -version

# Python 의존성 (Colab의 torch와 충돌 시 requirements.txt 의 torch 핀을 주석 처리)
!pip install -q -r requirements.txt

## 4. (데이터 수집용) Ollama 기동
수집 단계의 사양 추출/요구사항 증강은 OpenAI 호환 LLM 엔드포인트를 사용합니다.
Colab에서 Ollama를 백그라운드로 띄워 `qwen3:8b` 를 서빙합니다.

> 이미 만들어 둔 카탈로그(`parts_catalog.json`)를 Drive에서 재사용한다면 이 셀과 6번(수집)은 건너뛰어도 됩니다. 외부 API(DashScope 등)를 쓸 경우엔 5번에서 `LLM_*`만 바꾸고 이 셀을 건너뜁니다.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])   # 백그라운드 서버
time.sleep(5)
!ollama pull qwen3:8b

## 5. 시크릿 / .env 설정
Colab 좌측 🔑(보안 비밀)에 `MOUSER_API_KEY`(또는 Digi-Key 키)를 등록한 뒤 실행하세요.
경로는 Drive로 지정해 런타임 재시작에도 보존되게 합니다.

In [ ]:
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

env = f"""
MOUSER_API_KEY={secret('MOUSER_API_KEY')}
MOUSER_BASE=https://api.mouser.com
DIGIKEY_CLIENT_ID={secret('DIGIKEY_CLIENT_ID')}
DIGIKEY_CLIENT_SECRET={secret('DIGIKEY_CLIENT_SECRET')}
DIGIKEY_BASE=https://api.digikey.com

LLM_BASE_URL=http://localhost:11434/v1
LLM_API_KEY=ollama
LLM_MODEL=qwen3:8b

PDF_DIR={DRIVE_ROOT}/datasheets
CATALOG_PATH={DRIVE_ROOT}/parts_catalog.json
"""
with open('.env', 'w') as f:
    f.write(env)
print('.env 작성 완료')

## 6. 데이터 수집 (유통사 API + 데이터시트 파싱 + LLM 사양 추출)
→ `parts_catalog.json` 생성(Drive에 저장)

In [ ]:
!python -m pipeline.build_catalog --source mouser \
    --keywords-file pipeline/keywords.example.txt --limit 15

## 7. 학습 데이터 전처리
카탈로그 → TRL 대화 데이터셋(`train.jsonl`). `--augment` 는 Ollama로 요구사항을 다양화합니다.

In [ ]:
!python train/preprocess.py \
    --input-glob "$DRIVE_ROOT/parts_catalog.json" \
    --output "$DRIVE_ROOT/train.jsonl" \
    --augment --num-variations 5

## 8. QLoRA 학습 (Qwen3-8B)
어댑터는 Drive에 저장됩니다. T4(무료)에서 돌릴 경우 `--max-seq-len 1024 --batch-size 1` 로 낮추세요.

In [ ]:
!python train/train_qlora.py \
    --train-file "$DRIVE_ROOT/train.jsonl" \
    --output-dir "$DRIVE_ROOT/models/qwen3-8b-parts-lora"

## 9. 추론 테스트

In [ ]:
!python src/recommend.py \
    --adapter "$DRIVE_ROOT/models/qwen3-8b-parts-lora" \
    "5V를 3.3V로 변환하고 1A 이상 출력하는 LDO 추천해줘"